<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/sarvam2gguf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://huggingface.co/frankmorales2020/new-sarvam-30b-topo-governed-cbp-multi

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls -ltha /content/drive/MyDrive/topogpt/

In [4]:
!rm -rf /content/drive/MyDrive/topogpt/sarvam-30b-topo-governed-f16.gguf
!rm -rf /content/drive/MyDrive/topogpt/sarvam-30b-topo-governed-q8_0.gguf

In [5]:
import os
import requests
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import userdata

os.environ["HF_HUB_DISABLE_XET"] = "1"

MODEL_ID = 'sarvamai/sarvam-30b'
REPO_ID = 'frankmorales2020/new-sarvam-30b-topo-governed-cbp-multi'
LOCAL_DIR = './temp_merged_sarvam'
OUTPUT_DIR = '/content/drive/MyDrive/topogpt/merged_sarvam_topo_hf'
filename = 'certified_topological_best.pt'

print("=" * 70)
print("📦 PREPARING MERGED CHECKPOINT FOR GGUF CONVERSION")
print("=" * 70)

# 1. Download topological checkpoint via direct HTTPS stream (bypassing Xet 429 errors)
print("\n[1/3] Downloading certified topological checkpoint...")
url = f"https://huggingface.co/{REPO_ID}/resolve/main/{filename}"
token = userdata.get('HF_TOKEN')
headers = {"Authorization": f"Bearer {token}"}

response = requests.get(url, headers=headers, stream=True)
response.raise_for_status()

total_size = int(response.headers.get('content-length', 0))
block_size = 1024 * 1024  # 1 MB blocks
downloaded = 0

with open(filename, 'wb') as f:
    for chunk in response.iter_content(chunk_size=block_size):
        if chunk:
            f.write(chunk)
            downloaded += len(chunk)
            print(f"Downloaded: {downloaded / (1024**3):.2f} GB / {total_size / (1024**3):.2f} GB", end="\r")

print(f"\n✅ Successfully downloaded {filename}!")
checkpoint_path = os.path.abspath(filename)
checkpoint = torch.load(checkpoint_path, map_location="cpu")

# 2. Load base model and tokenizer
print("[2/3] Loading base model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

# [3/3] Integrating topological weights and saving locally first to avoid Drive sync errors
print("[3/3] Integrating topological weights and saving locally...")
if "model_state_dict" in checkpoint:
    base_model.load_state_dict(checkpoint["model_state_dict"], strict=False)

os.makedirs(LOCAL_DIR, exist_ok=True)

# Patch the model's modules to prevent the list attribute error during save
for _, submodule in base_model.named_modules():
    if hasattr(submodule, "_tied_weights_keys") and isinstance(submodule._tied_weights_keys, list):
        submodule._tied_weights_keys = {k: k for k in submodule._tied_weights_keys}

# Save locally with 2GB shards to ensure stable writes
base_model.save_pretrained(LOCAL_DIR, max_shard_size="2GB")
tokenizer.save_pretrained(LOCAL_DIR)

# Save the multi-task classifier heads separately as a companion artifact
heads_state = {
    "classifier_A": checkpoint.get("classifier_A", None),
    "classifier_B": checkpoint.get("classifier_B", None),
    "classifier_C": checkpoint.get("classifier_C", None),
    "state_hash": checkpoint.get("state_hash", None)
}
torch.save(heads_state, os.path.join(LOCAL_DIR, "multitask_classifier_heads.pt"))

print("🚚 Syncing completed shards safely to Google Drive...")
os.makedirs(OUTPUT_DIR, exist_ok=True)
!cp -r ./temp_merged_sarvam/* /content/drive/MyDrive/topogpt/merged_sarvam_topo_hf/

print(f"\n✅ Merged HF checkpoint saved and synced to: {OUTPUT_DIR}")

📦 PREPARING MERGED CHECKPOINT FOR GGUF CONVERSION

[1/3] Downloading certified topological checkpoint...
Downloaded: 18.45 GB / 18.45 GB
✅ Successfully downloaded certified_topological_best.pt!
[2/3] Loading base model and tokenizer...


config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

configuration_sarvam_moe.py:   0%|          | 0.00/3.96k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/sarvamai/sarvam-30b:
- configuration_sarvam_moe.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.13/dist-packages/transformers/configuration_utils.py:723: FutureWarning: `rope_scaling` is deprecated and will be removed in a future version. Set `rope_parameters` instead, with `rope_type` and `rope_theta` included.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.6MB            

... (10 outputs left out)

generation_config.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[3/3] Integrating topological weights and saving locally...


Writing model shards:   0%|          | 0/33 [00:00<?, ?it/s]

🚚 Syncing completed shards safely to Google Drive...

✅ Merged HF checkpoint saved and synced to: /content/drive/MyDrive/topogpt/merged_sarvam_topo_hf


In [ ]:
!ls -ltha /content/drive/MyDrive/topogpt/merged_sarvam_topo_hf

In [ ]:
!ls -lh /content/drive/MyDrive/topogpt/merged_sarvam_topo_hf/*.safetensors

In [ ]:
!python /content/drive/MyDrive/topogpt/llama.cpp/convert_hf_to_gguf.py /content/drive/MyDrive/topogpt/merged_sarvam_topo_hf \
    --outfile /content/drive/MyDrive/topogpt/sarvam-30b-topo-governed-f16.gguf \
    --outtype f16

In [7]:
!ls -ltha /content/drive/MyDrive/topogpt/

total 223G
drwx------ 17 root root 4.0K Nov  7 16:11 .
drwx------ 48 root root 4.0K Nov  7 16:10 ..
drwx------  2 root root 4.0K Nov  7 06:33 gguf_scratch
drwx------  2 root root 4.0K Nov  7 06:20 deepseek_r1_70b_topo_governed_merged
drwx------  2 root root 4.0K Nov  7 05:10 merged_model
drwx------  2 root root 4.0K Nov  7 04:10 base_backbone
-rw-------  1 root root 4.0G Nov  7 02:57 topo_cbp_trained_weights.safetensors
drwx------  2 root root 4.0K Nov  7 01:40 .cache
drwx------  2 root root 4.0K Nov  6 20:16 merged_sarvam_topo_hf
-rw-------  1 root root  80G Nov  4 06:49 mixtral_8x22b_q4_k_m.gguf
drwx------  2 root root 4.0K Nov  4 06:36 tmpdev
-rw-------  1 root root  29G Nov  4 02:30 mixtral_8x22b_q8_0-00004-of-00004.gguf
-rw-------  1 root root  38G Nov  4 00:40 mixtral_8x22b_q8_0-00003-of-00004.gguf
-rw-------  1 root root  38G Nov  3 21:46 mixtral_8x22b_q8_0-00002-of-00004.gguf
-rw-------  1 root root  37G Nov  3 18:55 mixtral_8x22b_q8_0-00001-of-00004.gguf
drwx------  2 root roo

In [4]:
!chmod a+x /content/drive/MyDrive/topogpt/tmpdev/build_quant/bin/llama-quantize

In [ ]:
!/content/drive/MyDrive/topogpt/tmpdev/build_quant/bin/llama-quantize

In [7]:
!ls -lh /content/drive/MyDrive/topogpt/sarvam-30b-topo-governed-f16.gguf

-rw------- 1 root root 2.4G Nov  6 20:27 /content/drive/MyDrive/topogpt/sarvam-30b-topo-governed-f16.gguf


In [6]:
!/content/drive/MyDrive/topogpt/tmpdev/build_quant/bin/llama-quantize /content/drive/MyDrive/topogpt/sarvam-30b-topo-governed-f16.gguf /content/drive/MyDrive/topogpt/sarvam-30b-topo-governed-Q4_K_M.gguf Q4_K_M

version: 0.8.0-dev (build 1, commit 5969566)
built with GNU 13.3.0 for Linux x86_64
llama_quantize: quantizing '/content/drive/MyDrive/topogpt/sarvam-30b-topo-governed-f16.gguf' to '/content/drive/MyDrive/topogpt/sarvam-30b-topo-governed-Q4_K_M.gguf' as Q4_K_M
llama_model_quantize: failed to quantize: tensor 'blk.1.ffn_down_exps.weight' data is not within the file bounds, model is corrupted or incomplete
llama_quantize: failed to quantize model from '/content/drive/MyDrive/topogpt/sarvam-30b-topo-governed-f16.gguf'


In [11]:
!chmod a+x /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-gguf

In [12]:
!/content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-gguf /content/drive/MyDrive/topogpt/sarvam-30b-topo-governed-f16.gguf

/content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-gguf: error while loading shared libraries: libggml-base.so.0: cannot open shared object file: No such file or directory


In [14]:
!export LD_LIBRARY_PATH=/content/drive/MyDrive/topogpt/llama.cpp/build/bin:$LD_LIBRARY_PATH
!/content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-gguf /content/drive/MyDrive/topogpt/sarvam-30b-topo-governed-f16.gguf

/content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-gguf: error while loading shared libraries: libggml-base.so.0: cannot open shared object file: No such file or directory


In [ ]:
!ls /content/drive/MyDrive/topogpt/llama.cpp/build/bin/

## cell1

In [9]:
import os

# Define paths
LLAMA_CPP_DIR = '/content/drive/MyDrive/topogpt/llama.cpp'
BUILD_BIN = os.path.join(LLAMA_CPP_DIR, 'build/bin')

# Set shared library path so llama binaries can find libggml.so and libllama.so
os.environ["LD_LIBRARY_PATH"] = f"{BUILD_BIN}:{os.environ.get('LD_LIBRARY_PATH', '')}"

print("✅ Environment paths and LD_LIBRARY_PATH configured!")

✅ Environment paths and LD_LIBRARY_PATH configured!


## cell 2

In [12]:
import os
import subprocess

MERGED_DIR = '/content/drive/MyDrive/topogpt/merged_sarvam_topo_hf'
LLAMA_CPP_DIR = '/content/drive/MyDrive/topogpt/llama.cpp'
LOCAL_Q8 = '/content/sarvam-30b-topo-governed-q8_0.gguf'
DRIVE_Q8 = '/content/drive/MyDrive/topogpt/sarvam-30b-topo-governed-q8_0.gguf'

print("=" * 70)
print("🚀 STEP 1 & 2: CONVERTING SHARDS DIRECTLY TO Q8_0 GGUF")
print("=" * 70)

# Run direct Q8_0 conversion on local SSD to avoid Drive IO bottlenecks
convert_cmd = [
    "python", os.path.join(LLAMA_CPP_DIR, "convert_hf_to_gguf.py"),
    MERGED_DIR,
    "--outfile", LOCAL_Q8,
    "--outtype", "q8_0"
]
subprocess.run(convert_cmd, check=True)

# Copy the completed file to Google Drive
print("\nCopying completed Q8_0 model to Google Drive...")
os.makedirs(os.path.dirname(DRIVE_Q8), exist_ok=True)
subprocess.run(["cp", LOCAL_Q8, DRIVE_Q8], check=True)

print(f"\n✅ Q8_0 model successfully generated and synced to:\n   {DRIVE_Q8}")

🚀 STEP 1 & 2: CONVERTING SHARDS DIRECTLY TO Q8_0 GGUF

Copying completed Q8_0 model to Google Drive...

✅ Q8_0 model successfully generated and synced to:
   /content/drive/MyDrive/topogpt/sarvam-30b-topo-governed-q8_0.gguf


## cell 3

In [14]:
!chmod a+x /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-gguf

In [15]:
import os
import subprocess

LLAMA_CPP_DIR = '/content/drive/MyDrive/topogpt/llama.cpp'
DRIVE_Q8 = '/content/drive/MyDrive/topogpt/sarvam-30b-topo-governed-q8_0.gguf'
inspect_bin = os.path.join(LLAMA_CPP_DIR, "build/bin/llama-gguf")

print("=" * 70)
print("🔍 STEP 3: INSPECTING GGUF TENSOR LAYOUT & HEADER INTEGRITY")
print("=" * 70)

inspect_cmd = [inspect_bin, DRIVE_Q8, "r", "n"]
subprocess.run(inspect_cmd, check=True)

print("\n✅ GGUF file integrity verified successfully!")

🔍 STEP 3: INSPECTING GGUF TENSOR LAYOUT & HEADER INTEGRITY

✅ GGUF file integrity verified successfully!


## cell 4

In [16]:
!chmod a+x /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-cli

In [ ]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu \
  --no-cache-dir

## inference-local

In [22]:
from llama_cpp import Llama

# Initialize the Llama model with GPU offloading (n_gpu_layers=-1 offloads all layers to your NVIDIA GPU)
model_path = '/content/drive/MyDrive/topogpt/sarvam-30b-topo-governed-q8_0.gguf'

llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,  # Offload all layers to GPU
    n_ctx=4096,       # Context window size
    verbose=False
)

# Run a test generation prompt
output = llm(
    "Explain the role of topology in neural network backpropagation:",
    max_tokens=256,
    temperature=0.7,
    stop=["</s>"]
)

print(output['choices'][0]['text'])

 In standard backprop, the gradient of the loss function with respect to the weights is computed using the chain rule. The chain rule is a fundamental concept in calculus that can be generalized to more abstract settings, such as topological spaces. In the context of neural networks, the topology of the space of functions (e.g., the space of continuous functions from <0xE2><0x84><0x9D>^n to <0xE2><0x84><0x9D>) is crucial for ensuring that the chain rule holds and that the gradient is well-defined. The chain rule is essentially a statement about the local behavior of the function, which is captured by its topology. In the context of neural networks, the topology of the space of functions is important for several reasons: (1) It ensures that the chain rule holds, which is necessary for computing the gradient. (2) It ensures that the gradient is well-defined, which is necessary for the optimization process. (3) It ensures that the loss function is continuous, which is necessary for the co

## hf

In [23]:
import os
import sys
from google.colab import userdata
from huggingface_hub import HfApi

# Target credentials and repo variables
HF_TOKEN = userdata.get('HF_TOKEN')
user_or_name = 'frankmorales2020'
repo_name = 'sarvam-30b-topo-governed-gguf'  # Adjust repo name if needed
repo_id = f"{user_or_name}/{repo_name}"

local_file = '/content/drive/MyDrive/topogpt/sarvam-30b-topo-governed-q8_0.gguf'
path_in_repo = 'sarvam-30b-topo-governed-q8_0.gguf'

print("=" * 70)
print(f"🚀 UPLOADING Q8_0 GGUF TO HUGGING FACE: {repo_id}")
print("=" * 70)

api = HfApi(token=HF_TOKEN)

# Ensure repository exists (creates it if it doesn't)
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

# Upload the large GGUF file with robust multi-part transfer
api.upload_file(
    path_or_fileobj=local_file,
    path_in_repo=path_in_repo,
    repo_id=repo_id,
    repo_type="model",
)

print(f"\n✅ Successfully uploaded to https://huggingface.co/{repo_id}")

🚀 UPLOADING Q8_0 GGUF TO HUGGING FACE: frankmorales2020/sarvam-30b-topo-governed-gguf


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...b-topo-governed-q8_0.gguf:   0%|          |  120MB / 34.2GB            

Validating                    :           |   0%            


✅ Successfully uploaded to https://huggingface.co/frankmorales2020/sarvam-30b-topo-governed-gguf


## inference-hf

In [24]:
from llama_cpp import Llama

# Load the GGUF model directly from Hugging Face Hub
llm = Llama.from_pretrained(
    repo_id="frankmorales2020/sarvam-30b-topo-governed-gguf",
    filename="sarvam-30b-topo-governed-q8_0.gguf",
    n_gpu_layers=-1,  # Offload all layers to your GPU
    n_ctx=4096,       # Context window
    verbose=False
)

# Run a test prompt
output = llm(
    "Explain the role of topology in neural network backpropagation:",
    max_tokens=256,
    temperature=0.7,
    stop=["</s>"]
)

print(output['choices'][0]['text'])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


./sarvam-30b-topo-governed-q8_0.gguf: reconstructing file:   0%|          |  0.00B / 34.2GB            

./sarvam-30b-topo-governed-q8_0.gguf: downloading bytes:           |  0.00B            

 In standard backprop, the gradient of the loss function with respect to the weights is computed using the chain rule. The chain rule is a fundamental concept in calculus that can be generalized to more abstract settings, such as topological spaces. In the context of neural networks, the topology of the space of functions (e.g., the space of continuous functions from <0xE2><0x84><0x9D>^n to <0xE2><0x84><0x9D>) is crucial for ensuring that the chain rule holds and that the gradient is well-defined. The chain rule is essentially a statement about the local behavior of the function, which is captured by its topology. In the context of neural networks, the topology of the space of functions is important for several reasons: (1) It ensures that the chain rule holds, which is necessary for computing the gradient. (2) It ensures that the gradient is well-defined, which is necessary for the optimization process. (3) It ensures that the loss function is continuous, which is necessary for the co